# 00 Data preparation (protocol Section 2)
Inputs in `MyDrive/BUSI_QB/raw/`:
* `Dataset_BUSI_with_GT.zip` (BUSI, required)
* `BUSI_issues/` with the supplementary file(s) of Pawlowska et al. 2023 (Mendeley doi:10.17632/k8t3gnx9h6.1)
* `BUSBRA.zip` (Zenodo doi:10.5281/zenodo.7730709)
* `BrEaST/` (optional; folder with `caseXXX.png` and `caseXXX_tumor.png`)

Outputs in `MyDrive/BUSI_QB/prepared/`: `busi_manifest_all.csv`, `busi_manifest.csv`, `busi_lesion.npz`, `busi_normal.npz`, `busbra.npz`, `breast.npz`, `checksums.json`, `prep_report.txt`. Send `prep_report.txt` before starting notebook 01.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, glob, zipfile, shutil, time
ROOT = '/content/drive/MyDrive/BUSI_QB'          # project folder on Drive
CODE = os.path.join(ROOT, 'code')                 # contains src/ (qsim.py, qbus.py, runner.py, prep.py) and protocols/
PREP = os.path.join(ROOT, 'prepared')
sys.path.insert(0, os.path.join(CODE, 'src'))
os.makedirs(PREP, exist_ok=True)
print(sorted(os.listdir(ROOT)))

In [ ]:
RAW = os.path.join(ROOT, 'raw'); WORK = '/content/raw'
os.makedirs(WORK, exist_ok=True)
for z in glob.glob(os.path.join(RAW, '*.zip')):
    dest = os.path.join(WORK, os.path.splitext(os.path.basename(z))[0])
    if not os.path.exists(dest):
        zipfile.ZipFile(z).extractall(dest)
for d in glob.glob(os.path.join(RAW, '*')):
    if os.path.isdir(d) and not os.path.exists(os.path.join(WORK, os.path.basename(d))):
        shutil.copytree(d, os.path.join(WORK, os.path.basename(d)))
print(os.listdir(WORK))

In [ ]:
import numpy as np, pandas as pd, prep
REPORT = []
def log(*a):
    s = ' '.join(str(x) for x in a); print(s); REPORT.append(s)
busi_root = [p for p in glob.glob(os.path.join(WORK, '*')) if 'busi' in p.lower() and 'issue' not in p.lower()][0]
files = prep.busi_files(busi_root)
log('BUSI images per class:', files.groupby('cls').size().to_dict())
log('images with more than one mask:', int((files.masks.map(len) > 1).sum()), '; without mask:', int((files.masks.map(len) == 0).sum()))

In [ ]:
issue_files = glob.glob(os.path.join(WORK, '*issue*', '**', '*.*'), recursive=True) + glob.glob(os.path.join(WORK, '*ssue*.*'))
log('issue files:', [os.path.basename(f) for f in issue_files])
groups, flags = prep.parse_issue_list(issue_files) if issue_files else ([], {})
for f in issue_files:   # show the structure so that the parsing can be checked
    if f.endswith('.csv'):
        print(pd.read_csv(f, sep=None, engine='python').head(8).to_string())
    elif f.endswith(('.xlsx', '.xls')):
        for sh, t in pd.read_excel(f, sheet_name=None).items():
            print(sh, t.shape); print(t.head(8).to_string())
from collections import Counter
log('duplicate rows:', len(groups), '; images in duplicate rows:', len(set().union(*groups)) if groups else 0)
log('flags:', dict(Counter(fl for v in flags.values() for fl in v)))
if not issue_files:
    log('DEVIATION: issue list not available; duplicates from the hash check only (protocol 2.1)')

In [ ]:
man, extra = prep.clean_busi(files, groups, flags, hash_thr=4)
if not issue_files:      # fallback: hash pairs define the duplicate groups
    man, extra = prep.clean_busi(files, [set(p) for p in extra], flags, hash_thr=4)
log('cleaning:', man.reason.replace('', 'kept').value_counts().to_dict())
log('kept per class:', man[man.keep == 1].groupby('cls').size().to_dict())
log('hash pairs not in the list (Hamming <= 4):', len(extra), extra[:20])
lesion = prep.assign_folds(man, extra, seed=2027)
log('folds x class:'); log(lesion.groupby(['fold', 'cls']).size().unstack().to_string())
for f in range(5):
    log(f'fold {f}: train', int((~lesion.fold.isin([f, (f + 1) % 5])).sum()), 'subsets 10/25/50%:',
        [int(lesion[f'sub{p:03d}_f{f}'].sum()) for p in (10, 25, 50)])
man.drop(columns=['path']).assign(masks=man.masks.map(lambda m: '|'.join(os.path.basename(x) for x in m))).to_csv(os.path.join(PREP, 'busi_manifest_all.csv'), index=False)
lesion.drop(columns=['path', 'masks']).to_csv(os.path.join(PREP, 'busi_manifest.csv'), index=False)

In [ ]:
sums = {}
X, Y = prep.arrays(list(lesion.path), list(lesion.masks))
np.savez_compressed(os.path.join(PREP, 'busi_lesion.npz'), x=X, y=Y, ids=lesion.image.values)
sums['busi_lesion_x'], sums['busi_lesion_y'] = prep.md5(X), prep.md5(Y)
log('BUSI lesion arrays', X.shape, 'mask fraction', round(float((Y > 0).mean()), 4), 'empty masks', int((Y.reshape(len(Y), -1).max(1) == 0).sum()))
nor = man[(man.keep == 1) & (man.cls == 'normal')]
Xn, _ = prep.arrays(list(nor.path), [[] for _ in range(len(nor))])
np.savez_compressed(os.path.join(PREP, 'busi_normal.npz'), x=Xn, ids=nor.image.values)
log('BUSI normal images (cleaned):', len(nor))

In [ ]:
bb = [p for p in glob.glob(os.path.join(WORK, '*')) if 'bra' in p.lower()]
if bb:
    t = prep.busbra(bb[0])
    Xe, Ye = prep.arrays(list(t.path), list(t.masks))
    np.savez_compressed(os.path.join(PREP, 'busbra.npz'), x=Xe, y=Ye, ids=t.image.values, patients=t.patient.values)
    sums['busbra_x'], sums['busbra_y'] = prep.md5(Xe), prep.md5(Ye)
    log('BUS-BRA images', len(t), 'patients', t.patient.nunique(), 'empty masks', int((Ye.reshape(len(Ye), -1).max(1) == 0).sum()))
else:
    log('BUS-BRA not found')
br = [p for p in glob.glob(os.path.join(WORK, '*')) if 'breast' in os.path.basename(p).lower()]
if br:
    t = prep.breast(br[0])
    Xe, Ye = prep.arrays(list(t.path), list(t.masks))
    np.savez_compressed(os.path.join(PREP, 'breast.npz'), x=Xe, y=Ye, ids=t.image.values, patients=t.patient.values)
    sums['breast_x'], sums['breast_y'] = prep.md5(Xe), prep.md5(Ye)
    log('BrEaST lesion images', len(t))
else:
    log('BrEaST not found (optional)')
json.dump(sums, open(os.path.join(PREP, 'checksums.json'), 'w'), indent=1)
open(os.path.join(PREP, 'prep_report.txt'), 'w').write('\n'.join(REPORT))

In [ ]:
# visual check: image, mask outline (green) for 6 BUSI and 3 BUS-BRA images
import matplotlib.pyplot as plt
from scipy import ndimage
def show(ax, x, y, title):
    ax.imshow(x, cmap='gray'); ax.contour(y > 0, [0.5], colors='lime', linewidths=0.8); ax.set_title(title, fontsize=8); ax.axis('off')
fig, axs = plt.subplots(1, 9, figsize=(18, 2.6))
for i, j in enumerate(np.linspace(0, len(X) - 1, 6).astype(int)):
    show(axs[i], X[j], Y[j], lesion.image[j])
if bb:
    e = np.load(os.path.join(PREP, 'busbra.npz'), allow_pickle=True)
    for i, j in enumerate(np.linspace(0, len(e['x']) - 1, 3).astype(int)):
        show(axs[6 + i], e['x'][j], e['y'][j], str(e['ids'][j]))
plt.tight_layout(); plt.savefig(os.path.join(PREP, 'check_masks.png'), dpi=110); plt.show()